# Publish Validated NYC Taxi Aggregates
Public destination only: hudaifaahmd/nyc-taxi-public-summary.
Import alongside notebooks 13, 12 and 09. Add as a dependent task after
08_monthly_pipeline_runner with Run if: All succeeded; no separate trigger.
Run only after repository contents/history are reviewed and public visibility
is approved. Reconciliation failures block writes. Unchanged reports skip
commits. No raw rows or credentials are published. No ETL or visibility change.
Export/evidence timestamps are not a daily Job heartbeat or real-time feed.
Secrets stay in nyc-taxi-publication/github-token. Never paste a token here.

In [0]:
%run ./13_prepare_public_summary

In [0]:
"""Validated summary promotion with explicit private and public entrypoints."""
import base64
from datetime import datetime, timezone
import json
import re
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import HTTPRedirectHandler, Request, build_opener

REPOSITORY = 'hudaifaahmd/nyc-taxi-public-summary'
REPO_API = '/repos/' + REPOSITORY
REPORT_API = REPO_API + '/contents/reports/snapshot.json'
STAGING_API = REPO_API + '/contents/_staging/snapshot.json'
MAX_BYTES = 262144


class PublishValidation(Exception):
    pass


class PublishHTTPError(Exception):
    def __init__(self,code):
        self.code=code
        super().__init__('GitHub request failed')


class PublicationNoRedirect(HTTPRedirectHandler):
    def redirect_request(self,req,fp,code,msg,headers,newurl):
        return None


def _pub_time(value):
    try:
        parsed=datetime.fromisoformat(value.replace('Z','+00:00'))
        if parsed.tzinfo is None: raise ValueError()
        return parsed.astimezone(timezone.utc)
    except Exception:
        raise PublishValidation('Invalid timestamp') from None


def validate_public_snapshot(snapshot):
    keys={'schemaVersion','verifiedOn','evidenceDate','evidenceThroughUtc',
          'generatedAtUtc','publicFeedConnected','publicationMode','months'}
    if not isinstance(snapshot,dict) or set(snapshot)!=keys:
        raise PublishValidation('Unexpected snapshot fields')
    if type(snapshot['schemaVersion']) is not int or snapshot['schemaVersion']!=1 or snapshot['publicFeedConnected'] is not False or snapshot['publicationMode']!='verified_snapshot':
        raise PublishValidation('Only the verified snapshot contract is supported')
    evidence=_pub_time(snapshot['evidenceThroughUtc'])
    generated=_pub_time(snapshot['generatedAtUtc'])
    if generated<evidence or snapshot['verifiedOn']!=evidence.date().isoformat() or snapshot['evidenceDate']!=evidence.date().isoformat():
        raise PublishValidation('Inconsistent evidence dates')
    months=snapshot['months']
    if not isinstance(months,list) or not 1<=len(months)<=1200:
        raise PublishValidation('Invalid monthly coverage')
    last=None
    for month in months:
        if not isinstance(month,dict) or set(month)!={'period','bronze','silver','gold','quarantine','amountCents','boundary'}:
            raise PublishValidation('Unexpected monthly fields')
        period=month['period']
        if not isinstance(period,str) or re.fullmatch(r'[0-9]{4}-(0[1-9]|1[0-2])',period) is None:
            raise PublishValidation('Invalid source period')
        ordinal=int(period[:4])*12+int(period[5:])
        if last is not None and ordinal!=last+1:
            raise PublishValidation('Coverage must be contiguous and ordered')
        last=ordinal
        for key in ['bronze','silver','gold','quarantine','boundary']:
            if type(month[key]) is not int or not 0<=month[key]<=9007199254740991:
                raise PublishValidation('Invalid count')
        if month['bronze']==0 or month['bronze']!=month['silver']+month['quarantine'] or month['silver']!=month['gold'] or month['boundary']>month['silver']:
            raise PublishValidation('Counts do not reconcile')
        cents=month['amountCents']
        if not isinstance(cents,str) or re.fullmatch(r'0|[1-9][0-9]{0,15}',cents) is None or int(cents)>9007199254740991:
            raise PublishValidation('Invalid amount cents')
    raw=json.dumps(snapshot,sort_keys=True,separators=(',',':'),allow_nan=False).encode()
    if len(raw)>MAX_BYTES: raise PublishValidation('Summary is too large')
    return raw


def _pub_file(api,path):
    try:
        record=api('GET',path)
    except PublishHTTPError as error:
        if error.code==404: return None
        raise
    if not isinstance(record,dict) or record.get('type')!='file' or record.get('encoding')!='base64' or not record.get('sha'):
        raise PublishValidation('Unexpected existing file')
    try:
        decoded=base64.b64decode(record['content'])
        if len(decoded)>MAX_BYTES: raise ValueError()
        snapshot=json.loads(decoded)
    except Exception:
        raise PublishValidation('Invalid existing summary encoding') from None
    return record,snapshot


def _pub_status(status,**fields):
    return dict(status=status,public_feed_enabled=False,scheduled_job_changed=False,**fields)


def _publish_summary(api,snapshot,*,expected_private):
    # Validate everything before the first write; stage and verify before promotion.
    promotion_started=False
    old=None
    read_report=None
    success_status='PRIVATE_SUMMARY_PUBLISHED' if expected_private else 'SUMMARY_PUBLISHED'
    try:
        raw=validate_public_snapshot(snapshot)
        repo=api('GET',REPO_API)
        if repo.get('private') is not expected_private or str(repo.get('full_name','')).lower()!=REPOSITORY:
            raise PublishValidation('Expected repository visibility not confirmed')
        branch=repo.get('default_branch')
        if branch!='main':
            raise PublishValidation('Expected main branch for public report delivery')
        suffix='?ref='+quote(branch,safe='')
        read_report=REPORT_API+suffix
        old=_pub_file(api,read_report)
        if old:
            validate_public_snapshot(old[1])
            previous=old[1]
            if (_pub_time(snapshot['evidenceThroughUtc'])<_pub_time(previous['evidenceThroughUtc']) or
                not {m['period'] for m in previous['months']}.issubset({m['period'] for m in snapshot['months']})):
                raise PublishValidation('Refusing stale or reduced coverage')
            if {k:v for k,v in snapshot.items() if k!='generatedAtUtc'}=={k:v for k,v in previous.items() if k!='generatedAtUtc'}:
                return _pub_status('UNCHANGED',report_updated=False)
        staged=_pub_file(api,STAGING_API+suffix)
        if staged: validate_public_snapshot(staged[1])
        def put(path,existing):
            body=dict(message='Update validated NYC Taxi summary',
                      branch=branch,content=base64.b64encode(raw).decode())
            if existing: body['sha']=existing[0]['sha']
            response=api('PUT',path,body)
            sha=response.get('content',{}).get('sha')
            if not sha: raise PublishValidation('Incomplete write confirmation')
            return sha
        staged_sha=put(STAGING_API,staged)
        confirmed=_pub_file(api,STAGING_API+suffix)
        if not confirmed or confirmed[0]['sha']!=staged_sha or confirmed[1]!=snapshot:
            raise PublishValidation('Staged readback mismatch')
        promotion_started=True
        report_sha=put(REPORT_API,old)
        confirmed=_pub_file(api,read_report)
        if not confirmed or confirmed[0]['sha']!=report_sha or confirmed[1]!=snapshot:
            raise PublishValidation('Report readback mismatch')
        return _pub_status(success_status,report_updated=True,
                           readback_verified=True,months=len(snapshot['months']),
                           report_path='reports/snapshot.json')
    except Exception as error:
        # A timeout may happen AFTER GitHub applied a write. Never falsely promise
        # that the previous file was retained; inspect it or report uncertainty.
        if promotion_started:
            try:
                current=_pub_file(api,read_report)
                if current and current[1]==snapshot:
                    return _pub_status(success_status,report_updated=True,
                                       readback_verified=True,months=len(snapshot['months']),
                                       report_path='reports/snapshot.json')
                if old and current and current[0]['sha']==old[0]['sha']:
                    return _pub_status('FAILED_PREVIOUS_RETAINED',report_updated=False)
                if old is None and current is None:
                    return _pub_status('FAILED_NO_REPORT_CREATED',report_updated=False)
            except Exception:
                pass
            return _pub_status('PUBLISH_STATE_UNKNOWN',report_updated=None)
        if isinstance(error,PublishValidation):
            return _pub_status('VALIDATION_BLOCKED',report_updated=False)
        if isinstance(error,PublishHTTPError):
            return _pub_status('FAILED_BEFORE_PROMOTION',report_updated=False,http_status=error.code)
        return _pub_status('FAILED_BEFORE_PROMOTION',report_updated=False)


def publish_private(api,snapshot):
    return _publish_summary(api,snapshot,expected_private=True)


def publish_public(api,snapshot):
    return _publish_summary(api,snapshot,expected_private=False)


def require_publication_success(result):
    if result.get('status') not in {'SUMMARY_PUBLISHED','UNCHANGED'}:
        raise RuntimeError('Summary publication did not complete; inspect the sanitized status above.') from None


def _run_publish(utilities,snapshot,*,expected_private):
    try:
        validate_public_snapshot(snapshot)
    except Exception:
        return _pub_status('VALIDATION_BLOCKED',report_updated=False)
    try:
        token=utilities.secrets.get(scope='nyc-taxi-publication',key='github-token')
        if not token or any(c.isspace() for c in token): raise ValueError()
    except Exception:
        return _pub_status('SECRET_READ_FAILED',report_updated=False)
    opener=build_opener(PublicationNoRedirect)
    def api(method,path,body=None):
        allowed=(method=='GET' and (path==REPO_API or
                 any(path.startswith(p+'?ref=') for p in (REPORT_API,STAGING_API)))) or (method=='PUT' and path in (REPORT_API,STAGING_API))
        if not allowed: raise PublishValidation('Unexpected request target')
        request=Request('https://api.github.com'+path,method=method,
                        data=None if body is None else json.dumps(body).encode(),
                        headers={'Authorization':'Bearer '+token,
                                 'Accept':'application/vnd.github+json',
                                 'Content-Type':'application/json',
                                 'X-GitHub-Api-Version':'2026-03-10',
                                 'User-Agent':'TaxiLakehousePrivatePublisher/1'})
        try:
            with opener.open(request,timeout=20) as response:
                data=response.read(MAX_BYTES*2+1)
                if len(data)>MAX_BYTES*2: raise PublishValidation('Response too large')
                return json.loads(data)
        except HTTPError as error:
            code=error.code
            error.close()
            raise PublishHTTPError(code) from None
    return _publish_summary(api,snapshot,expected_private=expected_private)


def run_private_publish(utilities,snapshot):
    return _run_publish(utilities,snapshot,expected_private=True)


def run_public_publish(utilities,snapshot):
    return _run_publish(utilities,snapshot,expected_private=False)


In [0]:
publication_result=run_public_publish(dbutils,public_snapshot)
print(json.dumps(publication_result,indent=2))
require_publication_success(publication_result)